In [0]:
RAW_PATH='/Volumes/workspace/default/healthcare_data_lake/'
BRONZE_PATH='/Volumes/workspace/default/healthcare_data_lake/bronze/'
SILVER_PATH='/Volumes/workspace/default/healthcare_data_lake/silver/'
GOLD_PATH='/Volumes/workspace/default/healthcare_data_lake/gold/'

#KEY OF GCP PROJECT
GCP_PROJECT='directed-truck-489818-t6'
BQ_QUERY='healthcare'
TEMP_GCS_BUCKET='healthcare-databricks-temp'


#secrets codes
GCP_SECRET_SCOPE='gcp-secret'
GCP_SECRET_KEY='gcp-sa-key'

In [0]:
silver_table=spark.read.format('delta').load(SILVER_PATH+'patients_clean')
silver_table.createOrReplaceTempView('silver_patients')
silver_labs = spark.read.format('delta').load(SILVER_PATH+'labs_clean')
silver_labs.createOrReplaceTempView('silver_labs')

In [0]:
from pyspark.sql.functions import sum
revenue_by_disease= spark.sql("""
                              select diagnosis_id, sum(treatment_cost) as total_revenue
                              from silver_patients
                              group by diagnosis_id
                              """)
revenue_by_disease.display()
revenue_by_disease.write.mode('overwrite').format('delta').save(GOLD_PATH+'revenue_by_disease')


diagnosis_id,total_revenue
10,155376.0
5,144149.0
3,149681.0
1,163926.0
2,104604.0
6,110654.0
9,63290.0
7,112219.0
4,103544.0
8,113539.0


In [0]:
recovery_rate=spark.sql("""
                        select diagnosis_id, count(*) as total_patients, 
                        sum(case when outcome_id=1 then 1 else 0 end) as recovered_patients, 
                        round(sum(
                            case when outcome_id=1 then 1 
                            else 0 
                            end)/total_patients,2) as recovery_rate
                        from silver_patients
                        group by diagnosis_id
                        """)
recovery_rate.display()
recovery_rate.write.mode('overwrite').format('delta').save(GOLD_PATH+'recovery_rate')



diagnosis_id,total_patients,recovered_patients,recovery_rate
10,26,13,0.5
5,23,7,0.3
3,24,9,0.38
1,28,5,0.18
2,17,5,0.29
6,20,4,0.2
9,10,3,0.3
7,18,5,0.28
4,17,8,0.47
8,17,4,0.24


In [0]:
avg_age_diagnosed=spark.sql("""
                   select diagnosis_id, round(avg(age),2) as avg_age
                   from silver_patients
                   group by diagnosis_id
                   """)
avg_age_diagnosed.display()
avg_age_diagnosed.write.mode('overwrite').format('delta').save(GOLD_PATH+'avg_age_diagnosed')

diagnosis_id,avg_age
10,60.23
5,62.22
3,60.13
1,58.86
2,60.65
6,59.25
9,57.7
7,61.89
4,55.12
8,54.88


In [0]:
high_risk_patients=spark.sql("""
                             select diagnosis_id, count(*) as high_risk_patients
                             from silver_patients
                             where age>60
                             group by diagnosis_id
                             """)
high_risk_patients.display()
high_risk_patients.write.mode("overwrite").save(GOLD_PATH+"high_risk_patients")

diagnosis_id,high_risk_patients
5,14
10,14
3,13
1,15
2,8
6,8
9,4
7,9
4,5
8,7


In [0]:
top_5_high_treament_cost=spark.sql("""
                                   select diagnosis_id, patient_id, sum(treatment_cost) as total_treatment_cost
                                    from silver_patients
                                    group by diagnosis_id, patient_id
                                    order by total_treatment_cost desc
                                    limit 5
                                    """)
top_5_high_treament_cost.display()
top_5_high_treament_cost.write.mode("overwrite").option("mergeSchema", "true").save(GOLD_PATH+"top_5_high_treament_cost")

                            

diagnosis_id,patient_id,total_treatment_cost
7,72,9973.0
3,192,9970.0
10,66,9970.0
8,26,9966.0
10,64,9833.0


In [0]:
diagnosis_admission_summary=spark.sql("""
                                      select diagnosis_id,
                                        count(*) as total_patients,
                                        min(admission_date) as first_admission_date,
                                        max(admission_date) as last_admission_date,
                                        max(discharge_date) as last_discharge_date,
                                        datediff(max(discharge_date), max(admission_date)) as latest_stay_days
                                      from silver_patients
                                      group by diagnosis_id
                                      order by last_admission_date desc
                                      """)
diagnosis_admission_summary.display()
diagnosis_admission_summary.write.mode("overwrite").save(GOLD_PATH+"diagnosis_admission_summary")

diagnosis_id,total_patients,first_admission_date,last_admission_date,last_discharge_date,latest_stay_days
1,28,2025-01-05,2025-06-30,2025-07-07,7
6,20,2025-01-08,2025-06-29,2025-07-04,5
10,26,2025-01-01,2025-06-27,2025-07-05,8
7,18,2025-01-05,2025-06-27,2025-07-09,12
5,23,2025-01-01,2025-06-26,2025-07-05,9
2,17,2025-01-25,2025-06-26,2025-07-08,12
4,17,2025-01-16,2025-06-23,2025-06-25,2
8,17,2025-01-09,2025-06-19,2025-07-10,21
3,24,2025-01-11,2025-06-16,2025-07-08,22
9,10,2025-01-07,2025-06-15,2025-06-21,6


In [0]:
lab_test_volume=spark.sql("""
                          select patient_id, count(*) as total_tests
                          from silver_labs
                          group by patient_id
                         """)
lab_test_volume.display()
lab_test_volume.write.mode("overwrite").save(GOLD_PATH+"lab_test_volume")

patient_id,total_tests
161,1
186,1
70,1
12,4
190,2
165,2
74,2
151,2
194,2
146,1


In [0]:
daily_lab_tests=spark.sql("""
                             select date(admission_date) as date, count(*) as daily_tests
                             from silver_patients
                             group by date(admission_date)
                             """)
daily_lab_tests.display()
daily_lab_tests.write.mode("overwrite").save(GOLD_PATH+"daily_lab_tests")



date,daily_tests
2025-02-08,2
2025-02-16,3
2025-04-11,2
2025-03-24,2
2025-03-17,2
2025-03-02,2
2025-03-16,2
2025-02-23,1
2025-01-10,1
2025-01-11,1
